# Unbalanced OT: Choice of Marginal Divergence

This notebook generates `fig:unbalanced-divergence-choice`.  The geometric part of the problem is kept fixed, and only the marginal divergence is changed in
$$
\min_{P\ge 0}\; \langle C,P\rangle
+\varepsilon\,\mathrm{KL}(P\mid a\otimes b)
+\tau D_\varphi(P\mathbf 1\mid a)
+\tau D_\varphi(P^\top\mathbf 1\mid b).
$$
The three panels compare KL, Burg (reverse KL), and total variation.  This isolates how the penalty on created and destroyed mass changes the shape of the transported marginals.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot
from matplotlib.colors import LinearSegmentedColormap
from scipy.optimize import minimize

from figure_style import (
    AXIS_LINE_WIDTH,
    BLUE,
    GRAY,
    RED,
    VIOLET,
    figure_dir,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

NAME = "unbalanced-divergence-choice"
OUT = figure_dir(NAME)

## Shared one-dimensional marginals

We reuse the Gaussian-mixture setting of `fig:unbalanced-mass-relaxation`: the source density $\alpha$ has two modes, while the target density $\beta$ has two nearby modes and a farther right mode.  The relaxation strength $\tau$ is fixed, so differences in the panels come only from the divergence $D_\varphi$.

In [ ]:
def gaussian(x, mean, sigma):
    return np.exp(-0.5 * ((x - mean) / sigma) ** 2) / (sigma * np.sqrt(2.0 * np.pi))


grid = np.linspace(-3.0, 3.0, 56)
dx = grid[1] - grid[0]

# A tiny uniform floor keeps the reverse-KL/Burg optimization well conditioned
# without altering the visible Gaussian-mixture structure.
alpha_density = 0.62 * gaussian(grid, -1.35, 0.30) + 0.38 * gaussian(grid, 0.35, 0.23) + 1.0e-3
beta_density = (
    0.30 * gaussian(grid, -0.80, 0.28)
    + 0.42 * gaussian(grid, 0.85, 0.32)
    + 0.28 * gaussian(grid, 2.20, 0.20)
    + 1.0e-3
)

alpha = alpha_density * dx
beta = beta_density * dx

cost = ot.dist(grid[:, None], grid[:, None], metric="sqeuclidean")
cost = cost / np.median(cost)
reference = np.outer(alpha, beta) + 1e-300

EPSILON = 0.025
TAU = 0.25
TV_SMOOTHING = 1.0e-4

## Three marginal penalties

For a transported marginal $r$ and a prescribed marginal $a$, we use
$$
D_{\mathrm{KL}}(r\mid a)=\sum_i r_i\log(r_i/a_i)-r_i+a_i,
\qquad
D_{\mathrm{Burg}}(r\mid a)=\sum_i a_i\log(a_i/r_i)-a_i+r_i,
$$
and a small smooth approximation of $D_{\mathrm{TV}}(r\mid a)=\sum_i |r_i-a_i|$ for differentiability.  The optimization is convex in the coupling $P$ and is small enough for a direct L-BFGS-B solve.

In [ ]:
def marginal_penalty_and_gradient(kind, marginal, target):
    if kind == "kl":
        value = np.sum(marginal * np.log(marginal / target) - marginal + target)
        gradient = np.log(marginal / target)
    elif kind == "burg":
        value = np.sum(target * np.log(target / marginal) - target + marginal)
        gradient = 1.0 - target / marginal
    elif kind == "tv":
        residual = marginal - target
        smooth_abs = np.sqrt(residual**2 + TV_SMOOTHING**2)
        value = np.sum(smooth_abs - TV_SMOOTHING)
        gradient = residual / smooth_abs
    else:
        raise ValueError(f"Unknown divergence kind: {kind}")
    return value, gradient


def initial_plan():
    plan = reference * np.exp(-cost / EPSILON)
    plan *= 0.75 * min(alpha.sum(), beta.sum()) / max(plan.sum(), 1e-30)
    return np.maximum(plan, 1e-14)


def solve_unbalanced(kind):
    n = len(alpha)
    start = initial_plan()

    def value_and_gradient(flat_plan):
        plan = flat_plan.reshape(n, n)
        row = plan.sum(axis=1)
        col = plan.sum(axis=0)

        value = np.sum(cost * plan) + EPSILON * np.sum(plan * np.log(plan / reference) - plan + reference)
        gradient = cost + EPSILON * np.log(plan / reference)

        row_value, row_gradient = marginal_penalty_and_gradient(kind, row, alpha)
        col_value, col_gradient = marginal_penalty_and_gradient(kind, col, beta)
        value += TAU * (row_value + col_value)
        gradient += TAU * row_gradient[:, None] + TAU * col_gradient[None, :]
        return float(value), gradient.ravel()

    result = minimize(
        lambda z: value_and_gradient(z),
        start.ravel(),
        jac=True,
        method="L-BFGS-B",
        bounds=[(1e-14, None)] * (n * n),
        options={"maxiter": 1200, "ftol": 1e-12, "gtol": 1e-7, "maxls": 50, "maxcor": 20},
    )
    return result.x.reshape(n, n)


plans = {
    "kl": solve_unbalanced("kl"),
    "burg": solve_unbalanced("burg"),
    "tv": solve_unbalanced("tv"),
}

## Coupling matrices with side marginals

Each panel uses the same visual convention as the KL relaxation figure: the central violet matrix shows the coupling, the left side compares $\alpha$ with $P\mathbf 1$, and the top side compares $\beta$ with $P^\top\mathbf 1$.  KL changes mass smoothly, Burg discourages vanishing transported marginals, and TV tends to keep a sharper active region.

In [ ]:
plan_cmap = LinearSegmentedColormap.from_list(
    "unbalanced_divergence_plan",
    ["#ffffff", "#f2e7f3", "#c9a3d5", VIOLET],
)

row_densities = {name: plan.sum(axis=1) / dx for name, plan in plans.items()}
col_densities = {name: plan.sum(axis=0) / dx for name, plan in plans.items()}
side_scale = 1.05 * max(
    alpha_density.max(),
    beta_density.max(),
    max(row.max() for row in row_densities.values()),
    max(col.max() for col in col_densities.values()),
)
plan_scale = max(np.sqrt((plan / dx**2).max()) for plan in plans.values())


def remove_side_axes(ax):
    remove_axes(ax)
    ax.set_facecolor("white")


def box_matrix_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(AXIS_LINE_WIDTH)
        spine.set_edgecolor("#333333")


def draw_panel(name, filename):
    plan = plans[name]
    row_density = row_densities[name]
    col_density = col_densities[name]
    plan_density = np.sqrt(plan / dx**2) / max(plan_scale, 1e-15)

    fig = plt.figure(figsize=(2.55, 2.35))
    gs = fig.add_gridspec(
        2,
        2,
        width_ratios=[0.30, 1.0],
        height_ratios=[0.30, 1.0],
        wspace=0.035,
        hspace=0.035,
    )
    ax_corner = fig.add_subplot(gs[0, 0])
    ax_top = fig.add_subplot(gs[0, 1])
    ax_left = fig.add_subplot(gs[1, 0])
    ax_matrix = fig.add_subplot(gs[1, 1])

    remove_side_axes(ax_corner)

    ax_top.fill_between(grid, col_density, beta_density, where=beta_density >= col_density, color=BLUE, alpha=0.18, linewidth=0)
    ax_top.fill_between(grid, 0, col_density, color=VIOLET, alpha=0.30, linewidth=0)
    ax_top.plot(grid, beta_density, color=BLUE, lw=0.85)
    ax_top.plot(grid, col_density, color=VIOLET, lw=0.8)
    ax_top.set_xlim(grid[0], grid[-1])
    ax_top.set_ylim(0, side_scale)
    remove_side_axes(ax_top)

    ax_left.fill_betweenx(grid, row_density, alpha_density, where=alpha_density >= row_density, color=RED, alpha=0.18, linewidth=0)
    ax_left.fill_betweenx(grid, 0, row_density, color=VIOLET, alpha=0.30, linewidth=0)
    ax_left.plot(alpha_density, grid, color=RED, lw=0.85)
    ax_left.plot(row_density, grid, color=VIOLET, lw=0.8)
    ax_left.set_ylim(grid[0], grid[-1])
    ax_left.set_xlim(side_scale, 0)
    remove_side_axes(ax_left)

    ax_matrix.imshow(
        plan_density,
        extent=[grid[0], grid[-1], grid[0], grid[-1]],
        origin="lower",
        cmap=plan_cmap,
        vmin=0,
        vmax=1,
        interpolation="nearest",
        aspect="auto",
    )
    ax_matrix.set_xlim(grid[0], grid[-1])
    ax_matrix.set_ylim(grid[0], grid[-1])
    box_matrix_axis(ax_matrix)

    save_pdf(fig, OUT / filename, pad_inches=0.045)
    plt.close(fig)


draw_panel("kl", "kl.pdf")
draw_panel("burg", "burg.pdf")
draw_panel("tv", "tv.pdf")